# Kaggriculture | Adaptive Farm Intelligence — Fieldbook Price-Aware Closeout

A compact Kaggriculture submission package.

## Method

The same three-day Fieldbook selector with one isolated closeout hypothesis: sell the final fertilizer sweep only when its visible market price is at least 2. All route tapes and shop-boundary choices are unchanged.

The notebook writes the required `submission.tar.gz`.

In [ ]:
# Exact submission archive with an integrity check.
import base64
import hashlib
import io
import tarfile
from pathlib import Path

ARCHIVE_B64 = (
    '...'
    'Tz8rP/8X3aZB/wAoBQA='
)
EXPECTED_SHA256 = "e3c2fe951dc2e2a5c54dfdd6abb8f4185925cfe2f6ed7e7f276c8f7edd6bf28b"
payload = base64.b64decode(ARCHIVE_B64)
assert hashlib.sha256(payload).hexdigest() == EXPECTED_SHA256
with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
    assert archive.getnames() == ["main.py"]
    source = archive.extractfile("main.py").read()
    compile(source, "main.py", "exec")
Path("submission.tar.gz").write_bytes(payload)
print("submission.tar.gz ready")
print("sha256:", EXPECTED_SHA256)
print("agent source:", len(source), "bytes")
